# Notebook 05 — Grad-CAM and Ablation Study
**Grad-CAM:** Spatial explainability — which regions drive the classification.
**Ablation:** Compare three model variants to justify design choices.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.config import (
    MODELS_DIR, PLOTS_DIR, RESULTS_DIR, OUTPUTS_DIR, SEED, CLASSES,
    ANOMALY_CLASSES, CLASS_TO_IDX, CLIP_LEN, JOINT_EPOCHS,
    LEARNING_RATE, BATCH_SIZE, NUM_CLASSES,
)
from src.model import (
    Encoder, Decoder, LSTMHead, TemporalPoolHead, JointModel,
    SingleFrameCNN, SingleFrameWrapper, freeze_decoder,
)
from src.dataset import (
    compute_class_weights, load_clip, PackedClipDataset,
)
from src.gradcam import compute_gradcam, overlay_heatmap, visualize_gradcam_batch
from src.evaluate import (
    compute_reconstruction_errors, find_optimal_threshold,
    evaluate_classifier, ablation_compare,
)
from src.train import pretrain_autoencoder, joint_train

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
CLIPS_DIR = OUTPUTS_DIR / 'clips_packed'
print(f'PyTorch: {torch.__version__}  Device: {DEVICE}')


## 1. Load Data and Best Joint Model

In [ ]:
# Load splits from packed meta.json
with open(CLIPS_DIR / 'meta.json') as f:
    meta = json.load(f)

train_npy = meta['train']['npy_paths']
y_train   = np.array(meta['train']['y'])
val_npy   = meta['val']['npy_paths']
y_val     = np.array(meta['val']['y'])
test_npy  = meta['test']['npy_paths']
y_test    = np.array(meta['test']['y'])
class_weights = compute_class_weights(y_train)

# Best joint model (LSTMHead + frozen encoder + class weights)
encoder = Encoder(); decoder = Decoder(); lstm_head = LSTMHead()
freeze_decoder(decoder)
joint_model = JointModel(encoder, decoder, lstm_head)
joint_model.load_state_dict(
    torch.load(MODELS_DIR / 'joint_model_best.pth', map_location=DEVICE))
joint_model = joint_model.to(DEVICE).eval()
print('Models loaded.')
print(f'Train clips: {len(train_npy)}  Val: {len(val_npy)}  Test: {len(test_npy)}')


## 2. Grad-CAM

### 2a. Single Example

In [ ]:
# Load one test clip from packed .npy for Grad-CAM
fight_idx = [i for i, l in enumerate(y_test) if l == CLASS_TO_IDX['Fighting']]

def _load_packed_as_6ch(npy_path):
    rgb = np.load(npy_path).astype(np.float32) / 255.0  # (T, H, W, 3)
    diffs = np.diff(rgb, axis=0)
    diffs = np.concatenate([np.zeros_like(rgb[:1]), diffs], axis=0)
    return np.concatenate([rgb, diffs], axis=-1)  # (T, H, W, 6)

clip = _load_packed_as_6ch(test_npy[fight_idx[0]])  # (T, H, W, 6)

heatmap, pred_cls, conf = compute_gradcam(
    encoder.to(DEVICE), lstm_head.to(DEVICE),
    clip, class_idx=CLASS_TO_IDX['Fighting'],
    frame_idx=CLIP_LEN // 2, device=DEVICE,
)

frame   = clip[CLIP_LEN // 2, :, :, :3]
overlay = overlay_heatmap(frame, heatmap, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(frame);               axes[0].set_title('Input Frame');    axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM');       axes[1].axis('off')
axes[2].imshow(overlay);             axes[2].axis('off')
axes[2].set_title(f'Overlay\nPred: {pred_cls} ({conf:.2f})')
plt.suptitle('Grad-CAM Example: Fighting', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gradcam_example.png', dpi=150, bbox_inches='tight')
plt.show()


### 2b. Batch Visualisation (2 examples per anomaly class)

In [ ]:
import numpy as _np

sample_clips = [_load_packed_as_6ch(test_npy[i]) for i in range(min(50, len(test_npy)))]
X_test_small = _np.stack(sample_clips)
y_test_small = y_test[:len(sample_clips)]

visualize_gradcam_batch(
    encoder.to(DEVICE), lstm_head.to(DEVICE),
    X_test_small, y_test_small, n_per_class=2,
    save_path=PLOTS_DIR / 'gradcam_batch.png', device=DEVICE,
)
plt.show()


## 3. Ablation Study

### Variant A: Single-frame CNN (no LSTM, no temporal context)

In [ ]:
print('=' * 55)
print('VARIANT A: Single-frame CNN (no temporal context)')
print('=' * 55)

sf_cnn    = SingleFrameCNN().to(DEVICE)
optimizer = torch.optim.Adam(sf_cnn.parameters(), lr=LEARNING_RATE)
weight_t  = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
ce_loss   = nn.CrossEntropyLoss(weight=weight_t)

from torch.utils.data import DataLoader

class _MidFrameDataset(PackedClipDataset):
    def __getitem__(self, idx):
        x, y = super().__getitem__(idx)
        return x[CLIP_LEN // 2, :3], y  # middle RGB frame (3, H, W)

tr_ds = DataLoader(_MidFrameDataset(train_npy, y_train, training=True),
                   batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
va_ds = DataLoader(_MidFrameDataset(val_npy, y_val, training=False),
                   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

best_sf_val = float('inf'); sf_patience = 0
for epoch in range(JOINT_EPOCHS):
    sf_cnn.train()
    for frames, labels in tr_ds:
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        ce_loss(sf_cnn(frames), labels).backward()
        optimizer.step()
    sf_cnn.eval(); va_loss = 0
    with torch.no_grad():
        for frames, labels in va_ds:
            va_loss += ce_loss(sf_cnn(frames.to(DEVICE)), labels.to(DEVICE)).item()
    if va_loss < best_sf_val:
        best_sf_val = va_loss; sf_patience = 0
        torch.save(sf_cnn.state_dict(), MODELS_DIR / 'ablation_sf.pth')
    else:
        sf_patience += 1
        if sf_patience >= 5: break

sf_cnn.load_state_dict(torch.load(MODELS_DIR / 'ablation_sf.pth', map_location=DEVICE))
sf_wrapper = SingleFrameWrapper(sf_cnn).to(DEVICE).eval()

y_pred_a, report_a = evaluate_classifier(sf_wrapper, test_npy, y_test, device=DEVICE)
sf_f1 = report_a['macro avg']['f1-score']
print(f'\nVariant A macro F1: {sf_f1:.4f}')


### Variant B: CNN+LSTM, UNFROZEN decoder

In [ ]:
print('=' * 55)
print('VARIANT B: CNN + TemporalPool (discards ordering)')
print('=' * 55)

enc_b = Encoder(); dec_b = Decoder(); head_b = TemporalPoolHead()
freeze_decoder(dec_b)
joint_b = JointModel(enc_b, dec_b, head_b)
enc_b.load_state_dict(torch.load(MODELS_DIR / 'encoder_pretrained.pth', map_location=DEVICE))
dec_b.load_state_dict(torch.load(MODELS_DIR / 'decoder_pretrained.pth', map_location=DEVICE))

hist_b = joint_train(joint_b, train_npy, y_train, val_npy, y_val,
                     device=DEVICE, class_weights=class_weights, use_packed=True)
torch.save(joint_b.state_dict(), MODELS_DIR / 'ablation_pool.pth')

y_pred_b, report_b = evaluate_classifier(joint_b.to(DEVICE), test_npy, y_test, device=DEVICE)
b_f1 = report_b['macro avg']['f1-score']
b_errors = compute_reconstruction_errors(joint_b, test_npy, device=DEVICE)
print(f'\nVariant B macro F1: {b_f1:.4f}')


### Variant C: CNN+LSTM, FROZEN decoder (our model)

In [ ]:
print('=' * 55)
print('VARIANT C: CNN + LSTM, frozen decoder (our model)')
print('=' * 55)

y_pred_c, report_c = evaluate_classifier(joint_model, test_npy, y_test, device=DEVICE)
c_f1 = report_c['macro avg']['f1-score']
c_errors = compute_reconstruction_errors(joint_model, test_npy, device=DEVICE)

from sklearn.metrics import roc_auc_score
b_auc = roc_auc_score((y_test != 0).astype(int), b_errors)
c_auc = roc_auc_score((y_test != 0).astype(int), c_errors)
print(f'\nVariant C macro F1: {c_f1:.4f}  Anomaly AUC: {c_auc:.4f}')


### Ablation Summary Table

In [ ]:
ablation = {
    "A: Single-frame CNN": {
        "Anomaly AUC": "N/A", "Classification F1": round(sf_f1, 4),
        "Notes": "No temporal context — middle frame only"},
    "B: CNN + TemporalPool": {
        "Anomaly AUC": round(b_auc, 4), "Classification F1": round(b_f1, 4),
        "Notes": "Discards temporal ordering via mean pooling"},
    "C: CNN + LSTM, frozen decoder (ours)": {
        "Anomaly AUC": round(c_auc, 4), "Classification F1": round(c_f1, 4),
        "Notes": "Preserves ordering; frozen decoder keeps anomaly signal"},
}

print("\n" + "=" * 75)
print(f"{'Model Variant':<38} {'AUC':>8} {'F1':>8}  Notes")
print("-" * 75)
for name, vals in ablation.items():
    print(f"  {name:<36} {str(vals['Anomaly AUC']):>8} {vals['Classification F1']:>8.4f}  {vals['Notes']}")
print("=" * 75)

with open(RESULTS_DIR / "ablation_results.json", "w") as f:
    json.dump(ablation, f, indent=2, default=str)


### Ablation Bar Chart

In [ ]:
variants  = ["A: Single-frame\nCNN", "B: CNN +\nTemporalPool", "C: CNN + LSTM\nFrozen (ours)"]
f1_scores = [sf_f1, b_f1, c_f1]
colors    = ["#AAAAAA", "#888888", "#1E88E5"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(variants, f1_scores, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylim(0, 1.1); ax.set_ylabel("Macro F1 Score")
ax.set_title("Ablation: Classification F1 by Model Variant", fontweight="bold")
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{score:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ablation_f1.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n[DONE] Grad-CAM and Ablation complete.")
print(f"Results: {RESULTS_DIR}  |  Plots: {PLOTS_DIR}")
